# 06 - RAG Evaluation with RAGAS

**Repo:** ai-learning-notes | **Folder:** rag-and-retrieval

## What this notebook covers

- Why RAG evaluation is harder than evaluating traditional software
- The four RAGAS metrics - what each measures and how each is computed
- Building a golden dataset that does not lie
- Real mistakes from local-rag-llm golden dataset and their RAGAS impact
- The NaN faithfulness problem with RAGAS 0.4.x and Claude
- Score pattern to root cause to fix
- CI/CD regression detection

**Pure Python only - no external dependencies.**

---
## 1. Why RAG Evaluation Is Different

For deterministic software: input -> expected output -> actual output -> pass/fail.

RAG breaks this in three ways:

**No single correct answer.** Many valid phrasings exist for any factual question.
String matching cannot work. You need a judge that understands meaning.

**Two failure modes.** The retriever can fail (wrong chunks) OR the generator can fail
(right chunks, wrong answer). You need metrics that tell you which one broke.

**Paraphrase matters.** 'SPIFFE IDs are used' and 'cryptographic identifiers like SPIFFE
are assigned' mean the same thing. String similarity marks the paraphrase as low quality.

RAGAS solves all three by using an LLM as the evaluator.

In [ ]:
import math

ground_truth = (
    "Zero Ambient Authority means an agent must never inherit "
    "the developers full ambient administrative privileges when executing a task."
)

rag_answers = [
    ("Correct paraphrase",
     "ZAA means agents should not be granted broad standing permissions "
     "from their parent process - only what is needed for the immediate task."),
    ("Verbatim correct", ground_truth),
    ("Wrong answer",
     "Zero Ambient Authority is a governance framework mandating regulatory "
     "approval before executing any AI actions."),
    ("Partial correct", "It means agents should not have full admin rights."),
]

def word_overlap(a, b):
    wa, wb = set(a.lower().split()), set(b.lower().split())
    if not wa or not wb:
        return 0.0
    return len(wa & wb) / len(wa | wb)

print("WHY STRING MATCHING FAILS")
print("=" * 60)
print(f"Ground truth: '{ground_truth[:60]}...'")
print()
print(f"{'Answer type':<22} {'String sim':>11}  Semantically OK?")
print("-" * 55)
for label, answer in rag_answers:
    sim = word_overlap(ground_truth, answer)
    ok = "YES" if "wrong" not in label.lower() else "NO"
    print(f"{label:<22} {sim:>11.3f}  {ok}")
print()
print("Correct paraphrase scores 0.25 - string match calls it poor quality.")
print("LLM-as-judge handles this correctly via semantic understanding.")

---
## 2. The Four RAGAS Metrics

```
               +------------------------------------------+
               |            RAG Pipeline                   |
               |                                           |
 Question ---->| Retriever -> Chunks -> LLM -> Answer      |
               |                                           |
               +------------------------------------------+
                      ^             ^           ^
                Ctx Precision  Ctx Recall   Faithfulness
                                            Ans Relevancy
```

**Context Recall** - Did retriever find all necessary chunks?
**Context Precision** - Are all retrieved chunks actually relevant?
**Faithfulness** - Does the answer stay within retrieved context? (hallucination check)
**Answer Relevancy** - Does the answer address the question asked?

In [ ]:
metrics = [
    {
        "name": "Context Recall",
        "formula": "#GT_sentences_supported / #GT_sentences_total",
        "how": "LLM checks each ground_truth sentence: supported by any retrieved chunk?",
        "low_means": "Retriever missed chunks - increase k or chunk_size",
        "local_rag_avg": 0.939,
    },
    {
        "name": "Context Precision",
        "formula": "#relevant_chunks / #total_retrieved_chunks",
        "how": "LLM judges each chunk: does it contribute to the ground truth answer?",
        "low_means": "Retriever returning noise - TOC chunks, off-topic sections",
        "local_rag_avg": 0.889,
    },
    {
        "name": "Faithfulness",
        "formula": "#answer_claims_in_context / #answer_claims_total",
        "how": "LLM extracts atomic claims from answer, verifies each against retrieved chunks",
        "low_means": "LLM hallucinated - added info not in the context. NaN = tool failure.",
        "local_rag_avg": 0.921,
    },
    {
        "name": "Answer Relevancy",
        "formula": "mean cosine_sim(original_question, synthetic_questions_from_answer)",
        "how": "LLM generates N questions from the answer, measures similarity to original Q",
        "low_means": "Answer is evasive, hedged, or off-topic",
        "local_rag_avg": 0.955,
    },
]

for m in metrics:
    print(f"{'='*60}")
    print(f"METRIC: {m['name']}  (local-rag-llm avg: {m['local_rag_avg']:.3f})")
    print(f"  Formula: {m['formula']}")
    print(f"  How:     {m['how']}")
    print(f"  Low:     {m['low_means']}")
print()
print("Faithfulness avg (0.921) is computed on 6/20 questions only.")
print("14/20 returned NaN due to RAGAS 0.4.x + Claude compatibility bug.")

In [ ]:
# Simulating context recall - the mechanism behind the score

def sim_context_recall(gt_sentences, retrieved_chunks):
    supported = 0
    results = []
    for sent in gt_sentences:
        sw = set(sent.lower().split())
        best = max(
            (len(sw & set(chunk.lower().split())) / len(sw) if sw else 0)
            for chunk in retrieved_chunks
        )
        ok = best > 0.4
        if ok:
            supported += 1
        results.append((sent, best, ok))
    return (supported / len(gt_sentences) if gt_sentences else 0), results

gt = [
    "Pillar 1 Infrastructure Networking ephemeral sandboxes and egress governance.",
    "Pillar 2 Data CMEK encryption mTLS and tenant partitioning.",
    "Pillar 3 Model treat system instructions as cryptographically attested artifacts.",
    "Pillar 4 Application Runtime LLM firewalls and Centralised Agent Gateways.",
    "Pillar 5 IAM SPIFFE IDs ABAC and JIT downscoping.",
    "Pillar 6 Observability Red Blue Green SecOps triad OpenTelemetry.",
    "Pillar 7 Governance Logic Reviews Risk-Stratified Attestation.",
]

small_chunks = [
    "Pillar 1 Infrastructure sandboxes. Pillar 2 Data CMEK mTLS partitioning.",
    "Table of contents: 7-Pillar Architecture page 9. Sandboxes page 13.",
    "Pillar 7 Governance Logic Reviews Attestation.",
]

large_chunks = [
    "Pillar 1 Infrastructure sandboxes egress. Pillar 2 Data CMEK mTLS partitioning.",
    "Pillar 3 Model attested artifacts. Pillar 4 Runtime LLM firewalls Gateways. Pillar 5 IAM SPIFFE ABAC JIT.",
    "Pillar 6 Observability Red Blue Green OpenTelemetry. Pillar 7 Governance Logic Reviews.",
]

print("CONTEXT RECALL SIMULATION - Q1 What are the 7 pillars?")
print("=" * 62)
for label, chunks in [("chunk_size=1000", small_chunks), ("chunk_size=2000", large_chunks)]:
    recall, results = sim_context_recall(gt, chunks)
    print(f"\n{label}: context_recall = {recall:.3f}")
    for sent, overlap, ok in results:
        print(f"  [{'YES' if ok else 'NO '}] overlap={overlap:.2f}  '{sent[:50]}'")
print()
print("Matches actual RAGAS results: 0.286 at chunk_size=1000, 0.857 at chunk_size=2000.")

---
## 3. Building a Good Golden Dataset

Three rules:

**Rule 1: Ground truths must be derivable from the document.**
If you write something the document does not say, RAGAS penalises retrieval for not finding it.
You get false negatives - the retriever looks bad when your ground truth is the problem.

**Rule 2: Comprehensive but not elaborated.**
Cover what the document says. Do not add your own explanations or inferences.

**Rule 3: No world knowledge beyond the document.**
If the comparison is not in the document - it is your knowledge, not the document's.
RAGAS cannot find it. Context recall suffers unfairly.

In [ ]:
issues = [
    {
        "question": "What is a Denial of Wallet attack?",
        "bad_sentence": (
            "Unlike a traditional Denial of Service attack targeting availability, "
            "DoW targets financial resources."
        ),
        "problem": "This DoS vs DoW comparison is NOT in the whitepaper - editorial elaboration.",
        "ragas_impact": "context_recall = 0.667 instead of 1.0",
        "fix": "Remove it. The whitepaper defines DoW but never compares it to DoS.",
    },
    {
        "question": "What is trajectory quality?",
        "bad_sentence": "Such an agent will fail unpredictably on similar tasks.",
        "problem": "This sentence does not appear anywhere in the whitepaper.",
        "ragas_impact": "context_recall = 0.500 instead of 1.0",
        "fix": "Remove it. Whitepaper says 'fragile success' - unpredictability inference is mine.",
    },
]

print("GOLDEN DATASET MISTAKES - real examples from local-rag-llm eval")
print("=" * 65)
for issue in issues:
    print(f"\nQuestion: '{issue['question']}'")
    print(f"Bad sentence:")
    print(f"  '{issue['bad_sentence'][:80]}'")
    print(f"Problem:  {issue['problem']}")
    print(f"Impact:   {issue['ragas_impact']}")
    print(f"Fix:      {issue['fix']}")
print()
print("After fixing both: avg context_recall improved from 0.912 to 0.939.")
print("Rule: every ground_truth sentence must trace to a specific document passage.")

---
## 4. The NaN Faithfulness Problem

In the local-rag-llm evaluation, faithfulness was NaN for 14 of 20 questions.
This is a known incompatibility between RAGAS 0.4.x and Claude as the judge LLM.

**How faithfulness is computed:**
1. RAGAS sends the answer to judge LLM: 'Extract all atomic factual claims as JSON'
2. LLM returns structured JSON list of claims
3. RAGAS verifies each claim against retrieved chunks
4. Score = supported_claims / total_claims

**Why NaN occurs:**
RAGAS 0.4.x expects a specific JSON schema.
Claude returns verbose markdown text instead of raw JSON.
Parser extracts 0 claims -> 0/0 -> NaN.

**NaN does NOT mean hallucination. It means the metric could not be computed.**

**Fix:** `pip install ragas==0.2.14` - older parser tolerates Claude's output format.

In [ ]:
eval_results = [
    ("Q1",  "7 pillars",          1.000, 0.250, 0.250, 0.857),
    ("Q2",  "Confused Deputy",    1.000, 0.989, 1.000, 1.000),
    ("Q3",  "Slopsquatting",      0.917, 0.971, 1.000, 1.000),
    ("Q4",  "ZAA/JIT",            None,  1.000, 1.000, 1.000),
    ("Q5",  "Vibe Diff",          None,  0.965, 1.000, 1.000),
    ("Q6",  "7 eval dims",        None,  1.000, 0.450, 1.000),
    ("Q7",  "DoW attack",         0.667, 0.937, 1.000, 0.667),
    ("Q8",  "MCP Spoofing",       1.000, 1.000, 1.000, 1.000),
    ("Q9",  "Intent Drift",       None,  0.978, 1.000, 1.000),
    ("Q10", "AgBOM",              1.000, 1.000, 1.000, 1.000),
    ("Q11", "Why eval different", None,  0.983, 0.500, 1.000),
    ("Q12", "Ephemeral sandbox",  None,  1.000, 1.000, 1.000),
    ("Q13", "Effective Trust",    1.000, 0.959, 1.000, 0.750),
    ("Q14", "Red/Blue/Green",     None,  0.945, 0.833, 1.000),
    ("Q15", "App vulns",          None,  0.987, 1.000, 1.000),
    ("Q16", "Repo poisoning",     0.778, 1.000, 1.000, 1.000),
    ("Q17", "Delegated identity", None,  0.910, 1.000, 1.000),
    ("Q18", "Session prefix",     None,  0.885, 0.750, 1.000),
    ("Q19", "Green Team",         1.000, 0.875, 1.000, 1.000),
    ("Q20", "Trajectory quality", 0.846, 0.986, 1.000, 0.500),
]

def avg_skip_none(vals):
    v = [x for x in vals if x is not None]
    return sum(v) / len(v) if v else 0.0

print("LOCAL-RAG-LLM RAGAS RESULTS - Clean Baseline (single document)")
print("=" * 68)
print(f"{'QID':<5} {'Topic':<22} {'Faith':>7} {'AnsRel':>7} {'CtxPrc':>7} {'CtxRec':>7}")
print("-" * 53)
for qid, topic, faith, relev, prec, recall in eval_results:
    f_str = f"{faith:.3f}" if faith is not None else "  NaN"
    flag = " <<" if (prec < 0.7 or recall < 0.85) else ""
    print(f"{qid:<5} {topic:<22} {f_str:>7} {relev:>7.3f} {prec:>7.3f} {recall:>7.3f}{flag}")
print("-" * 53)
avgs = [avg_skip_none([r[i] for r in eval_results]) for i in range(2, 6)]
print(f"{'AVG':<5} {'':22} {avgs[0]:>7.3f} {avgs[1]:>7.3f} {avgs[2]:>7.3f} {avgs[3]:>7.3f}")
nan_n = sum(1 for r in eval_results if r[2] is None)
print(f"NaN faithfulness: {nan_n}/20 (RAGAS 0.4.x + Claude bug - not a quality signal)")
print("<< = precision < 0.70 or recall < 0.85")

In [ ]:
# Regression detection - CI/CD threshold check

BASELINE   = {"avg_context_recall": 0.939, "avg_context_precision": 0.889, "avg_answer_relevancy": 0.955}
THRESHOLDS = {"avg_context_recall": 0.850, "avg_context_precision": 0.800, "avg_answer_relevancy": 0.900}

scenarios = [
    ("Good change - tuned junk filter",
     {"avg_context_recall": 0.952, "avg_context_precision": 0.901, "avg_answer_relevancy": 0.958}),
    ("Regression - bad junk filter removed real chunks",
     {"avg_context_recall": 0.812, "avg_context_precision": 0.934, "avg_answer_relevancy": 0.871}),
]

for name, results in scenarios:
    print(f"Scenario: {name}")
    print(f"{'Metric':<28} {'Baseline':>10} {'Current':>10} {'Threshold':>10} {'Status':>6}")
    print("-" * 68)
    failed = False
    for metric, current in results.items():
        baseline  = BASELINE[metric]
        threshold = THRESHOLDS[metric]
        delta     = current - baseline
        passes    = current >= threshold
        if not passes:
            failed = True
        status = "PASS" if passes else "FAIL"
        print(f"{metric:<28} {baseline:>10.3f} {current:>10.3f} {threshold:>10.3f} {status:>6}  ({delta:+.3f})")
    print(f"  -> {'CI FAILS - regression detected' if failed else 'CI PASSES - safe to merge'}")
    print()

---
## 5. Summary

| Concept | Key point |
|---|---|
| **Why RAGAS** | LLM-as-judge handles paraphrase - string matching cannot |
| **Context Recall** | Did retriever find all chunks? Low = chunk_size or k problem |
| **Context Precision** | Are retrieved chunks relevant? Low = noise or cross-doc contamination |
| **Faithfulness** | Did LLM stay in context? Low = hallucination. NaN = tool failure |
| **Answer Relevancy** | Does answer address question? Low = evasive or off-topic response |
| **Golden dataset** | Every ground_truth sentence must trace to a document passage |
| **Over-specification** | Adding your own knowledge creates false negatives in context_recall |
| **NaN faithfulness** | RAGAS 0.4.x + Claude bug - not a quality signal, skip or downgrade |
| **Diagnostic flow** | Low score -> inspect CSV retrieved_contexts -> root cause -> fix one thing |
| **CI/CD** | Gate on context_recall >= 0.85 per question and avg >= 0.90 |

**The evaluation insight:**
RAGAS is a diagnostic tool, not a score to maximise blindly.
Each metric points to a specific pipeline stage.
Low recall -> fix retrieval. Low precision -> fix noise. Low faithfulness -> fix prompting.
Always read the per-question CSV, not just the averages.

---
## Next Notebooks

- **07** - RAG security (prompt injection, 3-layer defence, fail-open vs fail-closed)
- **08** - Advanced RAG patterns (parent-document, multi-query, self-querying)
- **09** - Agentic RAG (tool use, multi-hop retrieval, memory)